# Phase II: Data Collection and Exploratory Data Analysis
##### INFO2950 Project by: Anderson Ramirez (ar2527) and Jackson Rhamy (jtr84)

### Research Question:

- What is the relationship between Joe Biden’s presidential approval rating and how often he is mentioned in political podcast episodes broadcast between January 20, 2021 and January 20, 2025?
    - Subquestion: Does this relationship between approval rating and mention rate vary by podcast political leaning? If so, how does it vary between conservative, moderate, and liberal podcasts?
    - Subquestion: Did the frequency of Biden mentions in political podcasts decline over the time of his presidency, regardless of countervailing changes in his approval rating? 

#### Package Imports

We will import Python libraries to manipulate and visualize our data.

In [1]:
import pandas as pd
import numpy as np
import duckdb

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

### Data Collection and Cleaning

#### President Biden's Approval Ratings

The Where and What of this dataset:
- Gallup Presidential Approval Ratings: Joe Biden, provided by Gallup.
- The dataset is can be downloaded as a CSV from the following link: https://news.gallup.com/poll/329384/presidential-approval-ratings-joe-biden.aspx

Why this dataset:
- This dataset provided information about Joe Biden's presidential approval rating over time, allowing us to explore how his approval fluctuated over the course of his presidency and compare this to how often he is mentoned in political podcasts.

##### Loading President Biden's Approval Ratings CSV Data

Here we are loading the data, printing its shape and DataFrame.

In [2]:
biden_ratings_df = pd.read_csv("bidenApprovalRating.csv")

print(f"The shape of biden_ratings_df is {biden_ratings_df.shape}")

biden_ratings_df.head()

##### Remove Headers, Unnecessary Columns, Rename Columns, and Convert Values

From the printed DataFrame, we can see that some rows have NaN values. These rows serve as headers that organize the data. For example, the first row specifies the unites, in this case percentages, for the corresponding columns, while `<b>2025</b>` is a heading for the data collected in 2025.

Since these rows only provide additional information about the data rather than actual observations, we will remove the rows with NaNs. We will also remove the `Disapprove` and `No opinion` columns because our research questions focus on the approval rating specifically. 

Next, we will rename the columns to make their labels more descriptive, and convert the percentage columns from an object to numeric data types to make sure they can be used for analysis. Since these values represent percentages, we will divide them by 100 to express them as a proportion between 0 and 1. This way the dataset will only have relevant observations with clear variable names and appropriate data types.

In [3]:
biden_ratings_df = biden_ratings_df.dropna().reset_index(drop=True)
biden_ratings_df = biden_ratings_df.drop(columns=['Disapprove', 'No opinion'])

biden_ratings_df = biden_ratings_df.rename(columns={'X.1':'Date', 'Approve':'Approval_Proportion'})
biden_ratings_df = biden_ratings_df.astype({'Approval_Proportion':float})
biden_ratings_df['Approval_Proportion'] = biden_ratings_df['Approval_Proportion'] / 100

biden_ratings_df.head()

In [4]:
# Creating intermediate saving point.
biden_ratings_df.to_csv('bidenApprovalRatingClean.csv', index=False)

##### Format the Date column

From the original printed DataFrame, we see that the dates are not consistently formatted, some months have duplicate observations, and two polling periods span multiple months. For example, October 2024 has two observations of .41 and .39, while all dates have polling ranges, like 2024 Oct 14-27, rather than standard monthly dates like 2024-10. Additionally, two polling periods cross month boundaries, 2022 Nov 9-Dec 2 and 2021 Jan 21-Feb 2.

- The first date issue we will tackle is the polling periods that span two months. To handle these cases, we will assign each polling observation to the month containing the midpoint of its polling period. This is consistent with the methodology explained by political scientists Will Jennings and Christopher Wlezien in a 2013 paper, where they use the midpoint between a survey's fieldwork start and end dates to find its associated date (see References section for more information). Rather than arbitrarily assigning a survey to its start or end date, this method uses the midpoint of the actual data collection period.
    - When doing replacing the 2 polling periods that span two months, we discovered that December 2022 does not have any data. In order to resolve this for now, we will be adding a row for December 2022 with NaN values.
- Next, we will address the presence of multiple observations within the same month. To resolve this, we will convert the Date column object type to a Datetime type, and then find the average approval rating for each month, that has multiple observations, using temporal aggregation, which combines higher frequency observations into a lower frequency, monthly dataset. Each poll is weighted equally regardless of its sample size or methodology; see our limitations section for more information. We chose this approach because our goal is to obtain one approval value per month for comparison with our podcast data, rather than develop a more sophisticated polling model.

In [5]:
# Loading intermediate data
biden_ratings_df = pd.read_csv('bidenApprovalRatingClean.csv')

biden_ratings_df['Date'] = biden_ratings_df['Date'].replace({'2021 Jan 21-Feb 2':'2021 Jan 27', '2022 Nov 9-Dec 2':'2022 Nov 20'})

biden_ratings_df = pd.concat([biden_ratings_df, pd.DataFrame({'Date': ['2022 Dec 20'], 'Approval_Proportion':[np.nan]})], ignore_index=True)

biden_ratings_df['Date'] = pd.to_datetime(biden_ratings_df['Date'].str.split('-').str[0]).dt.to_period('M').dt.to_timestamp()

biden_ratings_df = duckdb.sql('SELECT Date, AVG("Approval_Proportion") AS "Approval_Proportion", \
                                            FROM biden_ratings_df GROUP BY Date ORDER BY Date').df()

biden_ratings_df.head()

In [6]:
# Saving biden_ratings_df to a csv
biden_ratings_df.to_csv('bidenApprovalRatingClean.csv', index=False)

#### Political Podcast Data

The Where and What of this dataset:
- The dataset comes from the Brookings Institution’s Popular Political Podcast Dataset, which provides downloadable CSV data.
- It contains metadata for episodes of popular political podcasts, including show names, publication dates, titles, descriptions, and assigned political leanings.

Why this dataset:
- We use this dataset to measure how frequently Joe Biden is mentioned in political podcast episode metadata over time and examine how mention rates vary by political leaning, allowing us to compare these patterns with his presidential approval ratings.

##### Loading Political Podcast Data

Before cleaning our podcast dataset, we need to examine its structure to understand what information is available and find any potential issues that could affect our analysis. We will start by inspecting the dimensions, columns names, data types, and a sample of observations. This way we can see which columns are relevant and what cleaning steps are needed.

In [7]:
podcasts_df = pd.read_csv("podcast_full-dataset.csv")

print(f"The shape of podcasts_df is {podcasts_df.shape}")

print(podcasts_df.info())

podcasts_df.head()

##### Removing Unnecessary Columns and Handiling Missing values

From the initial inspection, we found that multiple columns are not necessary for answering our research questions. Since our analysis focuses on the frequency of Biden mentions in podcast metadata, we will only keep `Show Name`, `Full Date`, `Title`, `Description`, and `Partisan Leaning`. The remaining columns will be dropped because they do not contribute to our analysis.

We also saw that the `Description` column is missing 24 values. Instead of removing these 24 episodes, we will replace these values with empty strings. This way we can keep the observations and still find mentions of Biden via the episode titles, even if they have a missing description.

In [8]:
podcasts_df = podcasts_df.drop(columns=['Episode ID', 'Unnamed: 0', 'Category on Apple', 'Entered Dataset', 'Audio URL'])

podcasts_df['Description'] = podcasts_df['Description'].fillna('')

print(f"The shape of podcasts_df is {podcasts_df.shape}")

print(f"Missing values after cleaning: \n{podcasts_df.isna().sum()}")

# Creating intermediate saving point.
podcasts_df.to_csv('podcastDataCleaned.csv', index=False)

podcasts_df.head()

##### Find Duplicates

We want to check if the dataset has any duplicate episodes. Duplicate observatons can affect our results by causing certain epusodes to be counted more than once, potentially inflating Biden's mentions.

We will check for completely identical rows, and then if any episodes share the same `Show Name`, `Full Date`, and `Title`, as this combination might show duplicate episodes.

In [9]:
print(f"Duplicate rows: {podcasts_df.duplicated().sum()}")

print(f"Potential duplicate episodes: {podcasts_df.duplicated(subset=['Show Name', 'Full Date', 'Title']).sum()}")

The check found 144 completely identical rows and 192 potential duplicated episodes based on matching `Show Name`, `Full Date`, and `Title`. It is very possible that some episodes appear multiple times in the dataset. However, since the number of completely identical rows is lower, we will investigate these observations more before removing them.

In [10]:
duplicate_episodes = podcasts_df[podcasts_df.duplicated(subset=['Show Name', 'Full Date', 'Title'], keep=False)]

duplicate_episodes = duplicate_episodes.sort_values(by=['Show Name', 'Full Date', 'Title'])

duplicate_episodes.head(10)

##### Removing Duplicate Episodes

After seeing the potentially duplicated observations, we found that some episodes are completely identical across all five columns, while others share the same show, publication date, and the title but had different descriptions.

Since descriptions are important to finding Biden mentions, we decided to only remove the 144 identical rows. We keep the 48 episodes with different descriptions because they might represent different content, even if it shares the same title and date.

In [11]:
podcasts_df = podcasts_df.drop_duplicates().reset_index(drop=True)

print(f"Remaining duplicate rows: {podcasts_df.duplicated().sum()}")

print(f"The shape of podcasts_df now is {podcasts_df.shape}")

##### Filtering Episodes to Biden's Presidency 

Because our research focuses on podcast episodes published during Biden's presidency (January 20, 2021 to January 20, 2025), we will filter `podcasts_df` to include only episodes published within this time frame.

In the initial inspection, we saw that `Full Date` is stored as an object rather than a datetime data type. Thus, we will convert this column into a Pandas datetime object to conduct some chronological filtering.

We will then filter `podcasts_df` directly to keep only episodes published during Biden's presidency, rather than making a separate DataFrame. Episodes outside this time frame will be excluded because they are not relevant to our research questions. We will also reset the index of the filtered DataFrame so that observations are numbered consecutively. 

Finally, we will compare the dimensions of the dataset before and after filtering and make sure the publication dates match our desired time frame.

In [12]:
podcasts_df['Full Date'] = pd.to_datetime(podcasts_df['Full Date'])

og_shape = podcasts_df.shape
podcasts_df = podcasts_df[podcasts_df['Full Date'].between('2021-01-20', '2025-01-20')]

podcasts_df = podcasts_df.reset_index(drop=True)

print(f"Original dataset shape: {og_shape}")
print(f"Filtered dataset shape: {podcasts_df.shape}")

print(f"Earliest date: {podcasts_df['Full Date'].min()}")
print(f"Latest date: {podcasts_df['Full Date'].max()}")

# Creating intermediate saving point.
podcasts_df.to_csv('podcastDataCleaned.csv', index=False)

podcasts_df.head()

##### Find Biden Mentions in Podcast Metadata

To find which podcast episodes mention Biden, we will parse through the Title and Description columns using Pandas `str.contains()` method. 

First, we will check each column separately for the word "Biden," with case-insensitive matching to account for differences in capitalization and a regular expression to match the entire word. Then, we will combine results from both columns to create a binary variable called `Biden_mention`, where 1 means Biden is in the title or description and 0 indicates he is not. Episodes where Biden is mentioned in both columns will only be counted once.

In [13]:
title_biden = podcasts_df['Title'].str.contains(r"\bBiden\b", case=False, na=False)
description_biden = podcasts_df['Description'].str.contains(r"\bBiden\b", case=False, na=False)

podcasts_df['Biden_mention'] = (title_biden + description_biden).astype(int)

print(podcasts_df["Biden_mention"].value_counts())

podcasts_df[["Title", "Description", "Biden_mention"]].head()

##### Grouping Episodes by Month

To find the relationship between Biden's monthly approval rating and his mention frequency in political podcasts, we will organize the dataset by month.

We will create a `Month` column using the `Full Date` column to identify the year and month each episode was published. Then, we will group the episodes by month and calculate the total number of episodes published and the number of episodes mentioning Biden. These values will help us calculate the monthly Biden mention rate as the proportion of episodes mentioning Biden out of all episodes published that month. These results will be stored in a separate DataFrame called `monthly_podcasts_df` while preserving the original episode data.

In [14]:
podcasts_df["Month"] = podcasts_df["Full Date"].dt.to_period("M")

monthly_podcasts_df = podcasts_df.groupby("Month").agg(
    Total_episodes=("Biden_mention", "count"),
    Biden_episodes=("Biden_mention", "sum")
).reset_index()

monthly_podcasts_df["Mention_rate"] = np.round((monthly_podcasts_df["Biden_episodes"] / monthly_podcasts_df["Total_episodes"] * 100), 2)

# Creating intermediate saving point.
podcasts_df.to_csv('podcastDataCleaned.csv', index=False)
monthly_podcasts_df.to_csv('monthlyPodcastData.csv', index=False)

monthly_podcasts_df.head()

##### Inspect Political Leaning Column

We need to verify that our Partisan Leaning column is consistent because one of our research questions examines whether the relationship between Biden's approval rating and podcast mention frequency differs by political leaning.

We will see unique values in this column and count the number of observations belonging to each classification.

In [15]:
print(podcasts_df["Partisan Leaning"].unique())
print(podcasts_df["Partisan Leaning"].value_counts())

print(f"Missing values: {podcasts_df["Partisan Leaning"].isna().sum()}")

The `Partisan Leaning` column has four categories, `More Liberal`, `More Conservative`, `Moderate`, and `Unknown`. There are no missing values, so no additional cleaning is necessary. We will keep the `Unknown` category to keep all observations, but will be excluded when comparing political leanings.

### Combined Monthly Dataset

The Where and What of this dataset:
- This is a derived dataset created by merging Gallup’s Biden presidential approval ratings with monthly statistics from the Brookings political podcast dataset, producing 49 monthly observations and nine columns covering January 2021 through January 2025.

Why this dataset:
- We created this dataset to examine the relationship between Biden’s monthly presidential approval ratings and the proportion of political podcast episodes mentioning him, including how this relationship varies across liberal, moderate, and conservative podcasts.

##### Prepare the Datasets for Merging

Before merging the two datasets, we need to make sure the month columns use the same format and that their measurements are consistent.

In `biden_ratings_df`, the `Date` column represents the first day of each month, `Approval_Proportion` is already expressed as a proportion between 0 and 1. In `monthly_podcasts_df`, `Month` is stored as a monthly period, while `Mention_rate` is expressed as a percentage between 0 and 100.

We will make copies of both datasets to keep our previous cleaning work. Then, we will rename and convert the date columns into a common `Month` format. We will also recalculate the podcast mention rate as a proportion using the number of episodes that mention Biden divided by the total number of episodes.

Finally, we will conduct a sanity check for duplicate months in either dataset, since each month should only have one observation before we merge the data.

In [16]:
# Load Datasets
approval_monthly = pd.read_csv('bidenApprovalRatingClean.csv')
podcast_monthly = pd.read_csv('monthlyPodcastData.csv')

# Renames the approval date column and standardize both month columns to the first day of each month.
approval_monthly = approval_monthly.rename(columns={"Date": "Month"})

approval_monthly['Month'] = pd.to_datetime(approval_monthly['Month']).dt.to_period('M').dt.to_timestamp()

podcast_monthly['Month'] = pd.to_datetime(podcast_monthly['Month'].astype(str)).dt.to_period('M').dt.to_timestamp()

# Recalculate the podcast mention proportion from counts
podcast_monthly["Mention_Proportion"] = podcast_monthly["Biden_episodes"] / podcast_monthly["Total_episodes"]

# Keep only columns needed
approval_monthly = approval_monthly[["Month", "Approval_Proportion"]]
podcast_monthly = podcast_monthly[["Month", "Total_episodes", "Biden_episodes", "Mention_Proportion"]]

# Verify there is one row per month
print(f"Duplicate approval months: {approval_monthly["Month"].duplicated().sum()}")
print(f"Duplicate podcast months: {podcast_monthly["Month"].duplicated().sum()}")

print(approval_monthly.head())
print(podcast_monthly.head())

##### Merging the Datasets

Now that both datasets have been standardized, we can combine them using the `Month` column. We will use a left join with the podcast dataset as the primary table to keep all months with podcast observations, even if the corresponding approval ratings are missing.

The combined Dataset will have monthly approval proportion, total podcast episodes, episodes mentioning Biden, and the proportion of episodes mentioning Biden. 

To address our subquestion about political leaning, we will also use our cleaned episode level podcast data to calculate monthly Biden mention proportions for conservative, moderate, and liberal podcasts. These proportions will be added as separate columns to our combined dataset. allowing us to compare political leanings while keeping one observation per month.

After merging, we will check for missing values to find months without approval ratings or podcast information. We will also check wether the previously identified missing approval rating for December 2022 remains in the combined dataset.

In [17]:
# Merge monthly podcast data with approval ratings.
combined_monthly_df = duckdb.sql('SELECT p.Month, p.Total_episodes, p.Biden_episodes, p.Mention_Proportion, \
                                        a.Approval_Proportion FROM podcast_monthly AS p LEFT JOIN \
                                        approval_monthly AS a ON p.Month = a.Month ORDER BY p.Month').df()

# Loading the cleaned episode level podcast dataset
podcasts_leaning = pd.read_csv("podcastDataCleaned.csv")

# Standardize the Month column
podcasts_leaning["Month"] = pd.to_datetime(podcasts_leaning["Month"]).dt.to_period("M").dt.to_timestamp()

# Group episodes by month and political leaning
leaning_monthly = podcasts_leaning.groupby(["Month", "Partisan Leaning"]).agg(
    Total_episodes=("Biden_mention", "count"),
    Biden_episodes=("Biden_mention", "sum")
).reset_index()

# Calculate mention proportions for each political leaning
leaning_monthly["Mention_Proportion"] = leaning_monthly["Biden_episodes"] / leaning_monthly["Total_episodes"]

# Turn political leaning categories into separate columns
leaning_wide = leaning_monthly.pivot(index="Month", columns="Partisan Leaning", values="Mention_Proportion").reset_index()

# rename the political leaning columns
leaning_wide = leaning_wide.rename(columns={
    "More Conservative": "Conservative_Mention",
    "Moderate": "Moderate_Mention",
    "More Liberal": "Liberal_Mention",
    "Unknown": "Unknown_Mention"
})

# Add political leaning mention proportions to the combined data
combined_monthly_df = combined_monthly_df.merge(leaning_wide, on='Month', how='left')

# Check the shape of merged dataset
print(f"Combined dataset shape: {combined_monthly_df.shape}")

# Check for missing values and display them
print(combined_monthly_df.isna().sum())
print(combined_monthly_df[combined_monthly_df.isna().any(axis=1)])

combined_monthly_df.head()

In [18]:
# Creating intermediate saving point.
combined_monthly_df.to_csv('combinedMonthlyProportions.csv', index=False)

### Data Description



- **What are the observations (rows) and the attributes (columns)?**
The observations (rows) are individual months between January 2021 and January 2025, with each row summarizing podcast episode mentions of Joe Biden and his presidential approval rating for a given month.

The attributes (columns) are the following: 
*Month*: The month to which the observation corresponds.
*Total_episodes*: The total number of podcast episodes in the Brookings dataset for each month.
*Biden_episodes*: The number of podcast episodes in the Brookings dataset that mention Biden in their metadata each month.
*Mention_proportion*: The proportion of all podcast episodes in a given month that mention Biden, calculated by dividing Biden_episodes by Total_episodes.
*Approval_proportion*: Joe Biden's presidential approval rating for each month, expressed as a proportion.
*Moderate_Mention*: The proportion of podcast episodes labeled politically moderate that mention Biden in a given month.
*Conservative_Mention*: The proportion of podcast episodes labeled politically conservative that mention Biden in a given month.
*Liberal_Mention*: The proportion of podcast episodes labeled politically liberal that mention Biden in a given month.
*Unknown_Mention*: The proportion of podcast episodes with an unknown political leaning that mention Biden in a given month.

**- Why was this dataset created?**

This dataset was created to investigate the relationship between Joe Biden's presidential approval rating and the frequency that he was mentioned in political podcast episodes during his presidency, from January 20, 2021 until January 20, 2025. We combined the Gallup presidential approval rating data with podcast data from the Brookings Institution to determine whether changes in Biden's changes in public approval were at all associated with changes in how frequently he was mentioned in podcast metadata. Furthermore, the dataset lets us probe into how this relationship may have differed between liberal, moderate, and conservative podcasts, as well as whether Biden's mention rate declined over time independent of changes in approval rating.

**- How did you create this dataset? What tools, procedures, and data sources did you use?**

We created this dataset by combining two data sources: the "Gallup's Presidential Approval Ratings: Joe Biden" dataset and a Brookings Institution dataset containing information about episodes from the top 100 politcal podcasts. The Gallup dataset gave us Biden's presidential approval ratings over time, while the Brooking's dataset gave us access to podcast episode metadata, such as publication dates, titles, descriptions, and political readings. We downloaded both datasets as CSV files and used the Python libraries Pandas, NumPy, and DuckDB to clean, process, and combine the data.

First of all, we cleaned the Gallup dataset by removing unecessary rows and colums, converting approvate rating percentages to proportions, and standardizing the dates for ease of analysis. When multiple observations appeared in a single month, we calculated the average to get a single monthly approval rating. When the polling period spanned across 2 months, we assigned the observation to the month containing the midpoint of the polling period. Because December of 2022 had no approval observation, we preserved that month with a missing approval value instead of estimating one.

After this, we cleaned the Brookings Institution podcast dataset by removing unneeded columns, replacing missing descriptions with empty strings, and removing 144 completely duplicated rows. We kept episodes that shared a show name, publication, and title but have differing descriptions, since they could have different and important information that would be otherwise lost. Subsequently, we filtered the dataset down to episodes published between January 20, 2021, and January 20, 2025 (the time Biden's presidency). Using Pandas, we searched episode titles and descriptions for the word "Biden", without regard for capitalization, and created a binary variable that showed whether a given episode mentioned "Biden" in its metadata. 

To prepare the podcast dataset for merging with the approval rating dataset, we grouped podcast episodes by month and calculated the total amount of episodes published per month, number of episodes mentioning "Biden" per month, and from this got the proportion of Biden mentions per month. We repeated this process for each political leaning categories of liberal, moderate, conservative, and unknown.

After all of this cleaning and paring down of data, we were finally able to merge the two datasets. We merged the two on the 

- What processes might have influenced what data was observed and recorded and what was not?
- What preprocessing was done, and how did the data come to be in the form that you are using?
- If people are involved, were they aware of the data collection and if so, what purpose did they expect the data to be used for?
- Where can your raw source data be found, if applicable? Provide a link to the raw data (hosted on Github, in a Cornell Google Drive or Cornell Box).

### Data Limitations

There are a few limitations that may affect our findings.

First, our podcast metadata does not track every mention of Joe Biden in an episode. For example, an episode that discusses Biden extensively without mentioning his names in the description or title will not be counted. By contrast, a mention in the title or podcast description does not consequently necessitate that Biden be a major topic of discussion. Thus, the results we find will strictly be a measure of Biden’s presence in podcast metadata and not number of mentions in the audio.

Secondly, the data on podcasts derived from Brookings explicitly focuses on the top 100 political podcasts rather than the entire ecosystem of podcasts. As a result, the included podcasts may not fully represent every single political podcast.

Third, the data for presidential approval ratings are derived from surveys rather than measurements of every American's opinion. The uploaded file does not include sample sizes or margins of sampling error. Further it has minimal missing data. As a result, the approval ratings may not perfectly represent public opinion, and the lack of information about sampling uncertainty limits our ability to assess the precision of these estimates.

Fourth, the observational design of our project cannot establish causation. Although the project examines the relationship between Biden's approval rating and his frequency of mentions in political podcasts, both variables may be influenced by external factors, such as major political events, elections, and controversies. Thus, the analysis may be able to show associations between approval ratings and podcast mentions but cannot establish whether changes in one variable cause changes in the other. Nonetheless, because our research question focuses on correlation rather than causation, this limitation does not prevent us from addressing our primary research question.

Lastly, labels defining category and political ideology supplied by the Brookings Institute may simplify the actual views represented by a podcast. These classifications will be treated as provision source provided labels as opposed to perfect and nuanced measurements of a show’s political ideology.

### Exploratory data analysis

#### President Biden's Approval Ratings

In [19]:
# Loading cleaned Data.
biden_ratings_df = pd.read_csv('bidenApprovalRatingClean.csv', parse_dates=['Date'])

##### Descriptive statistics

Here we are going to look at the descriptive statistics from the cleaned dataset to beging analyzing the data.

In [20]:
biden_ratings_df.describe()

##### Findings

The cleaned Biden presidential approval dataset has 48 months with recorded observations. In these months, the mean approval rating was approximately 42.54%, while the median was 41%.

The approval ratings ranged from 36% to a maximum of 57%, with a standard deviation of approximately 5.67% points. The middle 50% of monthly approval ratings fell between 39% and 43%, showing that most observations were concentrated within a relatively small range. The mean being slightly bigger than the median shows that some higher approval observations might have pulled the average upward. 

The statistics give us an initial understanding of the distribution and variation in President Biden's monthly approval ratings, however, they do not show us how approval changed over time.

##### Monthly Approval Time Series

In [21]:
fig, time_series = plt.subplots(figsize=(12, 5))

time_series.plot(
    biden_ratings_df['Date'],
    biden_ratings_df['Approval_Proportion'],
    marker="o"
)

time_series.xaxis.set_major_locator(
    mdates.MonthLocator(bymonth=[1, 4, 7, 10])
)

time_series.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))

plt.xlabel('Date')
plt.ylabel('Approval Proportion')
plt.title("Biden's Monthly Presidential Approval Rating")
time_series.grid(axis='y', alpha=0.3)

plt.xticks(rotation=45)
plt.tight_layout()

plt.show()

##### Monthly Presidential Approval Ratings Over Time Findings

The time series plot shows Biden's monthly presidential approval ratings from January 2021 through January 2025. The graph shows a substantial decline in approval during 2021, approximately 57% in January to 43% by December. After this decline, approval ratings remained stable throughout the rest of his presidency, fluctuating between 37% and 44%. The lowest observed monthly approval rating occurred in July 2024, at 36%, followed by an increase to 43% in August 2024.

Overall, this visualization shows that Biden's approval ratings experienced a substantial decline early in his presidency, followed by a prolonged period of lower but relatively stable approval ratings. It is also worth noting that December 2022 contains missing values, resulting in a gap in the time-series plot. For now, we have chosen to preserve this gap rather than estimate or interpolate the missing approval rating, as doing so would introduce values that were not directly observed in the original dataset.

##### Histogram of Monthly Approval Ratings


In [22]:
biden_ratings_df.hist(
    column='Approval_Proportion',
    bins=[0.345, 0.365, 0.385, 0.405, 0.425, 0.445, 0.465, 0.485, 0.505, 0.525, 0.545, 0.565, 0.585],
    edgecolor='black',
    figsize=(9, 5)
)

plt.xlabel("Approval Proportion")
plt.ylabel("Number of Months")
plt.title("Distribution of Biden's Monthly Approval Ratings")
plt.show()

##### Histogram of Monthly Approval Ratings Findings

The histogram shows that Biden's monthly approval ratings are concentrated in a narrow range. Of the 48 months with a recorded approval rating, 40 fall between approximately .36 and .44, and the tallest bars (14 and 12 months) are around .39 and .42. There is a small, seperate group of 8 months with approval of .49 or higher, with a gap between about .45 and .48. This makes the distribution right-skewed, which explains why the mean is higher than the median. As seen in the time series plot, these high approval months are from early 2021.

#### Political Podcast Data

##### Line Plot of Numbers of Podcast Episodes Published per Month

In [23]:
monthly_podcasts_df.plot(
    x='Month',
    y='Total_episodes',
    kind='line',
    title='Total Podcast Episodes per Month',
    xlabel='Month',
    ylabel='Number of Episodes',
    legend=False,
    figsize=(9, 5)
)

The line plot shows that the total number of episodes generally increased throughout Biden's presidency, despite some monthly fluctuations. January 2021 and January 2025 have fewer episodes because our dataset only includes part of those months.

Since podcast publication frequency changes over time, using the proprotion of episodes mentioning Biden rather than counts will help account for these differences when examining mention frequency.

##### Line Plot of Biden Mention Rate Over Time

In [24]:
monthly_podcasts_df.plot(
    x='Month',
    y='Mention_rate',
    kind='line',
    title='Biden Mention Rate Over Time',
    xlabel='Month',
    ylabel='Percentage of Episodes Mentioning Biden',
    legend=False,
    figsize=(9, 5)
)

The line plot shows that Biden's mention rate fluctuayed throughout his presidency rather than consistently declining. Mention rates generally stayed between 20% and 35% for much of his presidency, wit ha increase in mid-2024 followed by a significant decline toward the end of 2024.

The graph shows a possible downward trend near the end of his presidency, but further analysis is needed to determine wether mention rates declined overall while accounting for changes in his approval rating.

##### Episodes and Shows by Political Leaning

In [25]:
print(f"Episodes by Political Leaning: {podcasts_df['Partisan Leaning'].value_counts()}")

print(f"Shows by Political Leaning: \n{podcasts_df.groupby('Partisan Leaning')['Show Name'].nunique()}")

Conservatives and liberal podcasts have a similar number of unique shows, but conservative podcasts account for more episodes (34,590 compared with 13,126). Moderate podcasts are less represented, with only 9 shows and 3,971 episodes.

##### Top 10 Podcasts by Episode Count

In [26]:
show_counts = podcasts_df.groupby('Show Name').size()
top_10_shows = show_counts.sort_values(ascending=False).head(10)
print(top_10_shows)

The results show that some podcast shows contribute more episodes than others. The Clay Travis and Buck Sexton Show has the most episodes (6,853), followed by Bill O'Reilly's No Spin News and Analysis (3,409) and Bannon's War Room (2,738). A small number of frequently publishing shows might have disproportionate influence on overall Biden's mention rates, since our analysis weights each episode equally.

##### Biden Mention Rate by Political Leaning

In [27]:
mention_by_leaning = podcasts_df.groupby('Partisan Leaning')['Biden_mention'].mean()
print(mention_by_leaning)

The results show that moderate podcasts have the highest Biden mention rate at .35, followed by conservative podcasts at .30 and liberal podcasts at .15. Podcasts with an unknown political leaning has the lowest mention rate at .11.

##### Biden Mention Rate Over Time by Political Leaning

In [28]:
monthly_leaning = podcasts_df.groupby(["Month", "Partisan Leaning"])["Biden_mention"].mean().unstack()

monthly_leaning[["More Conservative", "Moderate", "More Liberal"]].plot(
    kind='line',
    title='Biden Mention Rate Over Time by Political Leaning',
    xlabel='Month',
    ylabel='Proportion of Episodes Mentioning Biden',
    figsize=(9, 5)
)

The line plot shows that Biden's mention rates varied across political leanings throughout his presidency. Moderate podcasts had the highest mention rates from 2022 onward, while liberal podcasts tended to have the lowers. Conservative podcasts showed high mention rates earlier in the presidency. It is worth to mention that all groups had a sharp increase in mentions rates around mid-2024 followed by a decline.

#### Combined Monthly Dataset

In [29]:
# Load combined monthly data
combined_monthly_df = pd.read_csv('combinedMonthlyProportions.csv', parse_dates=['Month'])

##### Presidential Approval Rating and Biden Mention Rate Over Time

In [30]:
fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)

# Biden Approval Rating Plot
combined_monthly_df.plot(
    x='Month',
    y='Approval_Proportion',
    kind='line',
    ax=axes[0],
    title='Biden Presidential Approval Rating',
    ylabel='Approval Proportion',
    legend=False,
    figsize=(9, 5)
)

# Biden Podcast Mention Rate Plot
combined_monthly_df.plot(
    x='Month',
    y='Mention_Proportion',
    kind='line',
    ax=axes[1],
    title='Biden Podcast Mention Rate',
    ylabel='Mention Proportion',
    legend=False,
    figsize=(9, 5)
)

plt.tight_layout()
plt.show()

The plots show that Biden's approval rating declined during the beginning of his presidency before becoming stable around .4. Meanwhile, his podcast mention rate fluctuated throughout the period, including a sharp increase in mid-202 followed by a decline. The variables do not appear to move together.

##### Relationship Between Approval Rating and Biden Mention Rate

We will distinguish the beginning of Biden's presidency (2021) from later years to see whether the relationship between approval ratings and mention rates differs across these periods.

In [31]:
# Separate 2021 from later years
combined_monthly_df['Period'] = '2022-2025'

combined_monthly_df.loc[combined_monthly_df['Month'].dt.year == 2021, 'Period'] = '2021'

# Makes scatterplot
sns.scatterplot(
    data=combined_monthly_df,
    x='Approval_Proportion',
    y='Mention_Proportion',
    hue='Period'
)

plt.title('Biden Approval Rating vs. Podcast Mention Rate')
plt.xlabel('Approval Proportion')
plt.ylabel('Mention Proportion')
plt.legend(loc='upper left', bbox_to_anchor=(1, 1))
plt.show()

The scatterplot shows no clear relationship between Biden's approval rating and his podcast mention rate. In 2021, approval ratings were higher, while mention rates varied. From 2022 onward, approval ratings were concentrated at lower values, but mention rates continue to fluctuate.

##### Approval Rating vs. Biden Mention Rate by Political Leaning


In [32]:
# Scatterplot for conservative podcasts
sns.scatterplot(
    data=combined_monthly_df,
    x='Approval_Proportion',
    y='Conservative_Mention'
)

plt.title('Approval Rating vs. Conservative Podcast Mentions')
plt.xlabel('Approval Proportion')
plt.ylabel('Conservative Mention Proportion')
plt.show()

# Scatterplot for liberal podcasts
sns.scatterplot(
    data=combined_monthly_df,
    x='Approval_Proportion',
    y='Liberal_Mention'
)

plt.title('Approval Rating vs. Liberal Podcast Mentions')
plt.xlabel('Approval Proportion')
plt.ylabel('Liberal Mention Proportion')
plt.show()

# Scatterplot for Moderate_Mention
sns.scatterplot(
    data=combined_monthly_df,
    x='Approval_Proportion',
    y='Moderate_Mention'
)

plt.title('Approval Rating vs. Moderate Podcast Mentions')
plt.xlabel('Approval Proportion')
plt.ylabel('Moderate Mention Proportion')
plt.show()

The scatter shows no clear relationship between Biden's approval rating and his mention rate in either conservative or liberal podcasts. Conservative podcasts have a higher mention proportions than liberal podcasts across most approval ratings.

Both groups show variation in mention rates, suggesting that approval ratings alone might no explain how frequently podcasts mention Biden.

##### Correlation Between Approval Ratings and Mention Rates

We will calculate Pearson and Spearman correlation. Pearson correlation will measure the linear relationship, while Spearman will measure wether the two variables increase or decrease together. We are using Spearman, because our approval ratings are unevenly distributed, with higher values concentrated near the beginning of Biden's presidency.

In [33]:
correlation_df = combined_monthly_df[["Approval_Proportion", "Mention_Proportion", "Conservative_Mention", "Moderate_Mention", "Liberal_Mention"]]

print(f"Pearson Correlations:")
display(correlation_df.corr())

print(f"Spearman Correlations:")
display(correlation_df.corr(method='spearman'))

The Pearson and Spearman correlations show a weak positive relationship between Biden's approval rating and his overall podcast mention rate (0.195 and 0.097, respectively). 

The relationship varies by political leaning. Conservative podcasts show a weak positive correlation (Pearson: 0.315; Spearman: 0.289), while liberal podcasts show an even weaker postive correlation (0.254 and .085). Moderate podcasts show a stronger negative correlation (-.638 and -.512).

These findings add insight into the relationship between Biden's approval rating and podcast mention rates and how they might differ by political leaning.

##### Relationship Between Approval Rating and Time

We will calculate the correlation between Biden's monthly approval proportion and the number of months. This will also help- us find any potential multicollinearity when including both approval rating and time in our future regression models.

In [34]:
# This variable will count months since the beginning of the dataset
combined_monthly_df["Month_Number"] = range(1, len(combined_monthly_df) + 1)

combined_monthly_df[["Approval_Proportion", "Month_Number"]].corr()

The correlation between Biden's Approval rating and time is approximately -.721, showing a strong negative relationship. It suggests that Biden's approval rating decreased as his presidency progressed.

This strong correlation can also show potential multicollinearity when including both approval rating and time in our future regression models. We will need to look at this more using VIF to see if this is a concern.

### Preregistration of analyses

##### Hypothesis 1: 
Months in which Joe Biden's presidential approval rating was lower will tend to have a higher proportion of political podcast episodes mentioning him.

##### Analysis: 
We will run a regression using the monthly Biden mention rate as the outcome variable and monthly approval rating as the main explanatory variable. We will include a linear time trend to account for changes over the course of Biden's presidency and use Newey–West standard errors with three monthly lags to account for potential autocorrelation between consecutive months. We hypothesize that the coefficient for approval rating will be negative. We will report the coefficient, its uncertainty, and the results regardless of statistical significance.


##### Hypothesis 2: 
The relationship between Biden's approval rating and his frequency of mentions will differ across podcasts labeled liberal and conservative.

##### Analysis:
For each month, calculate the proportion of episodes mentioning Biden separately for conservative and liberal podcasts. Subtract the liberal mention rate from the conservative mention rate, then regress this difference on average monthly approval rating and a linear time trend. Use Newey–West standard errors with three monthly lags. We will test whether the approval-rating coefficient differs from zero, indicating that the relationship between approval and mention rates differs across the two political leanings.

### Questions for reviewers

* Are the proposed regression models and methods for accounting for time dependence appropriate for the relatively small number of monthly observations?
* Is assigning polling periods to months based on their fieldwork midpoint a reasonable approach, given that some surveys cross month boundaries?
* Are there additional variables or confounding factors, such as major political events or changes in publishing frequency, that should be accounted for in either analysis? If so, how would you suggest we account for this?

### Sources

To understand how to combat autocorrelation with Newey-West standard errors: https://metricgate.com/blogs/newey-west-standard-errors-explained/

https://www.geeksforgeeks.org/pandas/python-pandas-series-str-contains/

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=ffbb765c-73bd-48fb-b032-b98a4e6049ec' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>